In [13]:
import csv
import heapq

# Define the job seeker class
class JobSeeker:
    def __init__(self, skills, experience, salary, location, job_interest, sector, education_level, job_id):
        self.skills = skills
        self.experience = experience
        self.salary = salary
        self.location = location
        self.job_interest = job_interest
        self.sector = sector
        self.education_level = education_level
        self.job_id = job_id

    def __repr__(self):
        return (f"JobSeeker(job_id={self.job_id}, skills={self.skills}, "
                f"experience={self.experience}, salary={self.salary}, "
                f"location={self.location}, job_interest={self.job_interest}, "
                f"sector={self.sector}, education_level={self.education_level})")

# Define the job offer class
class JobOffer:
    def __init__(self, required_skills, min_experience, salary_range, location, sector, education_level):
        self.required_skills = required_skills
        self.min_experience = min_experience
        self.salary_range = salary_range
        self.location = location
        self.sector = sector
        self.education_level = education_level

    def __repr__(self):
        return (f"JobOffer(required_skills={self.required_skills}, "
                f"min_experience={self.min_experience}, "
                f"salary_range={self.salary_range}, "
                f"location={self.location}, "
                f"sector={self.sector}, "
                f"education_level={self.education_level})")

# Load job seekers from CSV
def load_job_seekers(filename):
    job_seekers = []
    try:
        with open(filename, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                job_seekers.append(JobSeeker(
                    skills=row['skills'].split(', '),
                    experience=int(row['experience']),
                    salary=int(row['salary']),
                    location=row['location'],
                    job_interest=row['job_interest'],
                    sector=row['sector'],
                    education_level=row['education_level'],
                    job_id=row['job_id']
                ))
        print(f"Loaded {len(job_seekers)} job seekers.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"Error: {e}")
    return job_seekers

# Calculate the cost of a job seeker for a specific job offer (based on mismatch)
def cost(job_seeker, job_offer):
    skill_mismatch = len(set(job_seeker.skills) - set(job_offer.required_skills))
    experience_mismatch = max(0, job_offer.min_experience - job_seeker.experience)
    salary_mismatch = max(0, job_offer.salary_range[0] - job_seeker.salary) + max(0, job_seeker.salary - job_offer.salary_range[1])
    location_mismatch = 0 if job_seeker.location == job_offer.location else 1
    return skill_mismatch + experience_mismatch + salary_mismatch + location_mismatch

# Improved heuristic to provide better matching for job seekers
def improved_heuristic(job_seeker, job_offer):
    # Skills mismatch: penalty for each missing skill
    missing_skills = set(job_offer.required_skills) - set(job_seeker.skills)
    skill_penalty = len(missing_skills) * 10  # Weight each missing skill with a penalty of 10
    
    # Experience mismatch: penalty for not having enough experience
    experience_penalty = max(0, job_offer.min_experience - job_seeker.experience) * 5  # Weight each year of missing experience with a penalty of 5
    
    # Salary mismatch: penalty if salary is outside the job offer's salary range
    salary_penalty = 0
    if job_seeker.salary < job_offer.salary_range[0]:
        salary_penalty = (job_offer.salary_range[0] - job_seeker.salary) / 1000  # Scale down large salary differences
    elif job_seeker.salary > job_offer.salary_range[1]:
        salary_penalty = (job_seeker.salary - job_offer.salary_range[1]) / 1000  # Scale down large salary differences
    
    # Location mismatch: penalty if the location doesn't match
    location_penalty = 0 if job_seeker.location.lower() == job_offer.location.lower() else 15  # Higher penalty for location mismatch
    
    # Education level mismatch: map education levels to numeric values for comparison
    education_levels = {
        "high school": 1,
        "associate's": 2,
        "bachelor's": 3, 
        "bachelor": 3,
        "master's": 4, 
        "master": 4,
        "phd": 5,
        "doctorate": 5
    }
    
    # Get numeric values for education levels (default to 0 if not found)
    seeker_edu_level = education_levels.get(job_seeker.education_level.lower(), 0)
    job_edu_level = education_levels.get(job_offer.education_level.lower(), 0)
    
    # Calculate education penalty - higher penalty if seeker's education is below required
    if seeker_edu_level < job_edu_level:
        education_penalty = (job_edu_level - seeker_edu_level) * 12  # Significant penalty for insufficient education
    else:
        education_penalty = max(0, (seeker_edu_level - job_edu_level) * 3)  # Small penalty for overqualification
    
    # Sector mismatch: penalty if sectors don't align
    # Exact match = 0 penalty, related sectors = medium penalty, unrelated = high penalty
    sector_penalty = 0
    if job_seeker.sector.lower() != job_offer.sector.lower():
        # Define sector relationships (simplified example - could be expanded)
        related_sectors = {
            "tech": ["information technology", "software", "it"],
            "information technology": ["tech", "software", "it"],
            "business": ["finance", "marketing", "consulting"],
            "finance": ["business", "banking", "accounting"],
            "marketing": ["business", "advertising", "sales"],
            "healthcare": ["medical", "pharma", "health"],
        }
        
        # Check if sectors are related
        if job_offer.sector.lower() in related_sectors.get(job_seeker.sector.lower(), []) or \
           job_seeker.sector.lower() in related_sectors.get(job_offer.sector.lower(), []):
            sector_penalty = 8  # Medium penalty for related sectors
        else:
            sector_penalty = 18  # High penalty for unrelated sectors
    
    # Calculate job interest alignment (if job_interest matches job offer's sector)
    interest_penalty = 0
    if job_seeker.job_interest.lower() != job_offer.sector.lower():
        interest_penalty = 5  # Penalty if job seeker's interest doesn't align with job sector
    
    # Combine all penalties into the final heuristic value
    total_penalty = (
        skill_penalty + 
        experience_penalty + 
        salary_penalty + 
        location_penalty + 
        education_penalty + 
        sector_penalty +
        interest_penalty
    )
    
    return total_penalty


# A* search algorithm with the improved heuristic
def a_star_search(job_seekers, job_offer, top_n=5):
    # Use a list to store all matches with their heuristic values
    matches = []
    
    # Find the maximum possible penalty to calculate normalized scores later
    max_penalty = 0
    for job_seeker in job_seekers:
        penalty = improved_heuristic(job_seeker, job_offer)
        max_penalty = max(max_penalty, penalty)
    
    # Add a small value to max_penalty to avoid division by zero
    if max_penalty == 0:
        max_penalty = 1
    
    # Evaluate each job seeker using the heuristic
    for job_seeker in job_seekers:
        seeker_penalty = improved_heuristic(job_seeker, job_offer)
        
        # Calculate compatibility score (0-100%)
        # Lower penalty means higher compatibility
        compatibility_score = 100 * (1 - seeker_penalty / max_penalty)
        
        # Ensure score is within 0-100 range
        compatibility_score = max(0, min(100, compatibility_score))
        
        matches.append((seeker_penalty, compatibility_score, job_seeker))
    
    # Sort the matches by their heuristic values (lowest penalty first)
    sorted_matches = sorted(matches, key=lambda x: x[0])
    
    # Return the top N matches (with compatibility scores)
    top_matches = [(match[1], match[2]) for match in sorted_matches[:top_n]]
    
    return top_matches




# Define sample job offers for testing
def define_sample_jobs():
    job1 = JobOffer(required_skills=["Copywriting"], min_experience=3, salary_range=(50000, 70000), location="Algiers", sector="Tech", education_level="Bachelor")
    job2 = JobOffer(required_skills=["Java", "Data Analysis"], min_experience=2, salary_range=(40000, 60000), location="Tizi ouzou", sector="Tech", education_level="Master")
    job3 = JobOffer(
            required_skills=["SEO", "Google Ads"],
            min_experience=3,
            salary_range=(50000, 90000),
            location="Algiers",
            sector="Business",
            education_level="Master's"
        )
    job4 = JobOffer(
            required_skills=["Java", "SQL"],
            min_experience=2,
            salary_range=(40000, 80000),
            location="Tizi Ouzou",
            sector="Information Technology",
            education_level="Bachelor's"
        )
    return [job1, job2, job3, job4]


# Main function to test the implementation
def main():
    job_seekers = load_job_seekers("job_seekers_with_ids.csv")
    
    if not job_seekers:
        print("No job seekers found. Check the file.")
        return  # Exit the function if no job seekers were loaded
    
    job_offers = define_sample_jobs()

    # Test A* search on each job offer and print the top 5 best matches
    for i, job_offer in enumerate(job_offers):
        print(f"\nJob Offer #{i+1}: {job_offer}")
        top_matches = a_star_search(job_seekers, job_offer, top_n=5)
        
        if top_matches:
            print(f"Top 5 best matches:")
            for j, (compatibility_score, job_seeker) in enumerate(top_matches):
                print(f"{j+1}. Job Seeker ID: {job_seeker.job_id}")
                print(f"   Compatibility Score: {compatibility_score:.2f}%")
                print(f"   Skills: {job_seeker.skills}")
                print(f"   Experience: {job_seeker.experience} years")
                print(f"   Salary: {job_seeker.salary}")
                print(f"   Location: {job_seeker.location}")
                print(f"   Sector: {job_seeker.sector}")
                print(f"   Education: {job_seeker.education_level}")
                print()
        else:
            print("No suitable matches found.")



if __name__ == "__main__":
    main()


Loaded 10000 job seekers.

Job Offer #1: JobOffer(required_skills=['Copywriting'], min_experience=3, salary_range=(50000, 70000), location=Algiers, sector=Tech, education_level=Bachelor)
Top 5 best matches:
1. Job Seeker ID: 13305
   Compatibility Score: 89.54%
   Skills: ['Copywriting', 'Analytics', 'SEO', 'Google Ads', 'Social Media']
   Experience: 5 years
   Salary: 50899
   Location: Algiers
   Sector: Business
   Education: Bachelor's

2. Job Seeker ID: 17873
   Compatibility Score: 89.54%
   Skills: ['Node.js', 'Responsive Design', 'React', 'CSS', 'HTML', 'WordPress']
   Experience: 3 years
   Salary: 64278
   Location: Algiers
   Sector: Information Technology
   Education: Bachelor's

3. Job Seeker ID: 19411
   Compatibility Score: 89.54%
   Skills: ['CSS', 'Node.js', 'JavaScript', 'Responsive Design', 'WordPress']
   Experience: 9 years
   Salary: 57971
   Location: Algiers
   Sector: Information Technology
   Education: Bachelor's

4. Job Seeker ID: 15077
   Compatibility Sc